In [12]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
import pickle

with open("/content/drive/MyDrive/binary_dataset.pkl", "rb") as f:
    data = pickle.load(f)

X_train = data["X_train"]
X_test = data["X_test"]
y_train = data["y_train"]
y_test = data["y_test"]

client_data = data["client_data"]
poisoned_client_data = data["poisoned_client_data"]

NUM_CLIENTS = data["NUM_CLIENTS"]
malicious_ids = data["malicious_ids"]

print("Loaded successfully")
print("Number of malicious clients:", len(malicious_ids))
print("First 10 malicious IDs:", malicious_ids[:10])

Loaded successfully
Number of malicious clients: 50
First 10 malicious IDs: [327, 57, 12, 379, 140, 125, 114, 71, 377, 52]


In [20]:
import random
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.metrics import accuracy_score

In [21]:
def create_binary_model():

    model = Sequential([
        Dense(64,activation='relu',input_shape=(X_train.shape[1],)),
        Dense(32,activation='relu'),
        Dense(1,activation='sigmoid')
    ])

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model


global_model = create_binary_model()

global_weights = global_model.get_weights()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [22]:
detected_attackers = []

for cid,(X_client,y_client) in enumerate(poisoned_client_data):

    attack_ratio = np.mean(y_client)

    if attack_ratio > 0.95 or attack_ratio < 0.05:
        detected_attackers.append(cid)

print("Detected attackers:",len(detected_attackers))
print(detected_attackers[:20])

Detected attackers: 500
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]


In [23]:
trusted_clients=[]

for cid,client in enumerate(poisoned_client_data):

    if cid not in detected_attackers:
        trusted_clients.append(client)

print("Trusted Clients :",len(trusted_clients))

Trusted Clients : 0


In [24]:
print("Number of clients:", len(client_data))
print("Number of poisoned clients:", len(poisoned_client_data))
print("Number of malicious IDs:", len(malicious_ids))
print("First 10 malicious IDs:", malicious_ids[:10])


Number of clients: 500
Number of poisoned clients: 500
Number of malicious IDs: 50
First 10 malicious IDs: [327, 57, 12, 379, 140, 125, 114, 71, 377, 52]


In [ ]:
# Reputation-based client exclusion, aligned with Algorithm 2 in the manuscript.
# The defender does NOT know the true malicious IDs.  Instead, clients whose
# label distribution falls outside the plausible attack-ratio band
# [0.05, 0.95] are flagged as suspect and excluded from aggregation.

trusted_clients = []

for cid, client in enumerate(poisoned_client_data):
    if cid not in detected_attackers:
        trusted_clients.append(client)

# Detection quality against the (unknown) ground truth, for diagnostic
# purposes only.  Not used by the aggregation logic itself.

true_positives  = len(set(detected_attackers) & set(malicious_ids))
false_positives = len(set(detected_attackers) - set(malicious_ids))
false_negatives = len(set(malicious_ids) - set(detected_attackers))

print("Trusted Clients :", len(trusted_clients))
print("Excluded Clients:", len(detected_attackers))
print("Detection TP    :", true_positives)
print("Detection FP    :", false_positives)
print("Detection FN    :", false_negatives)


In [26]:
NUM_ROUNDS = 10

CLIENTS_PER_ROUND = 50

history=[]

global_weights = global_model.get_weights()

In [27]:
for rnd in range(NUM_ROUNDS):

    print("\nRound",rnd+1)

    selected_clients=random.sample(
        trusted_clients,
        CLIENTS_PER_ROUND
    )

    local_weights=[]

    for X_client,y_client in selected_clients:

        local_model=create_binary_model()

        local_model.set_weights(global_weights)

        local_model.fit(
            X_client,
            y_client,
            epochs=1,
            batch_size=32,
            verbose=0
        )

        local_weights.append(local_model.get_weights())

    new_weights=[]

    for weights in zip(*local_weights):

        new_weights.append(
            np.mean(weights,axis=0)
        )

    global_weights=new_weights

    global_model.set_weights(global_weights)

    pred=(global_model.predict(X_test,verbose=0)>0.5).astype(int)

    acc=accuracy_score(y_test,pred)

    history.append(acc)

    print("Accuracy =",acc)


Round 1
Accuracy = 0.9847486512597289

Round 2


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Accuracy = 0.9868297198931243

Round 3


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Accuracy = 0.9878237325563184

Round 4


...

Accuracy = 0.990185481832519

Round 9


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Accuracy = 0.9903979307168521

Round 10


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Accuracy = 0.9903002352444945


In [28]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix

pred=(global_model.predict(X_test)>0.5).astype(int)

acc=accuracy_score(y_test,pred)
pre=precision_score(y_test,pred)
rec=recall_score(y_test,pred)
f1=f1_score(y_test,pred)

print("Accuracy :",acc)
print("Precision:",pre)
print("Recall   :",rec)
print("F1 Score :",f1)

print(confusion_matrix(y_test,pred))

20152/20152 ━━━━━━━━━━━━━━━━━━━━ 21s 1ms/step
Accuracy : 0.9903002352444945
Precision: 0.9978308164627933
Recall   : 0.9922218110139919
F1 Score : 0.9950184091881082
[[ 13922   1358]
 [  4897 624684]]


In [29]:
global_model.save("/content/drive/MyDrive/BIAB_secure_model.keras")

print("Secure FL Model Saved")

Secure FL Model Saved


In [30]:
import pandas as pd

comparison = pd.DataFrame({

    "Method":[
        "Baseline FL",
        "Label Flipping Attack",
        "BiAB-IoT Defense"
    ],

    "Accuracy":[
        0.9899,
        0.9883,
        acc
    ],

    "Precision":[
        0.9994,
        0.9996,
        pre
    ],

    "Recall":[
        0.9903,
        0.9884,
        rec
    ],

    "F1":[
        0.9948,
        0.9940,
        f1
    ]

})

comparison

,Method,Accuracy,Precision,Recall,F1
0,Baseline FL,0.9899,0.999400,0.990300,0.994800
1,Label Flipping Attack,0.9883,0.999600,0.988400,0.994000
2,BiAB-IoT Defense,0.9903,0.997831,0.992222,0.995018
